In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import healpy as hp
from scipy.odr import Model, Data, ODR
from pygdsm import gsm16
import pysm3 as pysm
import pysm3.units as u
import cmasher as cmr
import wget
from pathlib import Path
import zipfile 
import gdown

In [ ]:
def sform(param, xxx):
    '''linear reggression'''
    return xxx * param[0] + param[1]

def remove_off(map, hasmap, maperr, haserr):
    nside_superpix=8

    super_pixs=  np.arange(hp.nside2npix(nside_superpix )) #added
    offsets = np.zeros_like(super_pixs)*1. 
    offmap = np.full(hp.nside2npix(nside_superpix), np.nan)  
   
    nansize =0 
    for jj, ipix in enumerate(super_pixs) : 

        super_map = np.zeros(hp.nside2npix(nside_superpix))
        super_map [ipix] =1
        patch =  (hp.ud_grade(super_map, nside_out=hp.get_nside(map))) .astype('bool')
     
        try:

            errx = haserr[patch] * hasmap[patch]
            erry =  maperr * map[patch]
            mydata = Data(hasmap[patch], map[patch], wd=errx**-2, we=erry**-2)
            myodr = ODR(mydata, Model(sform), beta0=[0.0, 0.0])
            myoutput = myodr.run()
            cfit = myoutput.beta[1]
            if np.corrcoef(hasmap[patch], map[patch])[0,1] > 0.75:
                offsets[jj] = cfit
            else:
                offsets[jj] = np.nan
        except  ValueError: 
            nansize+=1 
            offsets[jj]= np.nan 
            
    offmap[super_pixs] = offsets 

    avec = np.nanmedian( offsets )
    
    return avec

input_dir = Path("../input_data")
beta_path = input_dir / "final_beta_1deg.npy"
cs_path   = input_dir / "final_cs_1deg.npy"                                   
  
if not (beta_path.exists() and cs_path.exists()):                             
    input_dir.mkdir(parents=True, exist_ok=True)
    zip_path = input_dir / "synchrotron_spectral_parameters.zip"              
                                                                                
    gdown.download(
        id="11N8u-Vrd3stFNN_v6UirGjLbcYrJm5GC",                               
        output=str(zip_path),                                                 
        quiet=False,
    )                                                                         
                  
    with zipfile.ZipFile(zip_path) as zf:                                     
        zf.extractall(input_dir)
                                                                                
beta_map = np.load(beta_path)
cs_map   = np.load(cs_path)

nside = 256

# Show average spectral index value
move_nu = np.arange(45, 2300, 10)
move_beta = np.array([beta_map + cs_map * np.log(move_nu[xx] / 45) for xx in range(len(move_nu))])
plt.figure(figsize=(10, 5))
plt.plot(move_nu, np.mean(move_beta, axis=1))
plt.fill_between(move_nu, np.mean(move_beta, axis=1) - np.std(move_beta, axis=1), np.mean(move_beta, axis=1) + np.std(move_beta, axis=1), alpha=0.3)
plt.xlabel('Frequency (MHz)')
plt.ylabel('Average Spectral Index')
plt.title('Average Spectral Index vs Frequency')
plt.show()

move_nu = np.arange(0.045, 70, 0.10)
move_beta = np.array([beta_map + cs_map * np.log(move_nu[xx] / 0.045) for xx in range(len(move_nu))])
plt.figure(figsize=(10, 5))
plt.plot(move_nu, np.mean(move_beta, axis=1))
plt.fill_between(move_nu, np.mean(move_beta, axis=1) - np.std(move_beta, axis=1), np.mean(move_beta, axis=1) + np.std(move_beta, axis=1), alpha=0.3)
plt.xlabel('Frequency (GHz)')
plt.ylabel('Average Spectral Index')
plt.title('Average Spectral Index vs Frequency')
plt.show()


In [ ]:
def rotate_map(mapin, coord_in , coord_out ) : 
    alm = hp.map2alm (mapin )
    R=  hp.Rotator(coord=[coord_in, coord_out ])
    alm =  R.rotate_alm (alm )
    map_out = hp.alm2map (alm , nside= hp.get_nside(mapin ))
    return map_out

#empirical data set to nside 256
fstring="haslam408_ds_Remazeilles2014.fits"

try :
    print(f"reading {fstring} ")
    hasmap_=  hp.read_map(f"../input_data/{fstring}" ,h=True  )  
except FileNotFoundError: 
    filename = wget.download( f"https://lambda.gsfc.nasa.gov/data/foregrounds/haslam_2014/{fstring}", out ="../input_data")
    hasmap =  hp.read_map(f"../input_data/{fstring}"  )

hasmap = hp.ud_grade(hasmap, nside)
hasmap = hasmap - 8.9

#read in the ff maps
fstring="COM_CompMap_freefree-commander_0256_R2.00.fits"
try :
    print(f"reading {fstring} ")
    tmp =  hp.read_map(f"../input_data/{fstring}" , field= ['EM_ML', 'TEMP_ML']) 
except FileNotFoundError: 
    filename = wget.download( f"https://lambda.gsfc.nasa.gov/data/foregrounds/haslam_2014/{fstring}", out ="../input_data")
    tmp =  hp.read_map(f"../input_data/{fstring}", field= ['EM_ML', 'TEMP_ML'])
planck_te = tmp [1] #in K

fstring = "EM_mean_std.fits"
try :
    print(f"reading {fstring} ")
    hust=  hp.read_map(filename=f"../input_data/{fstring}")  
except FileNotFoundError: 
    filename = wget.download( f"https://zenodo.org/records/10523170/files/{fstring}" , out ="../input_data")
    hust =  hp.read_map(f"../input_data/{fstring}")

Tff = lambda Te , nu,EM  : Te * (1.0 - np.exp(-tauff(Te,nu, EM ))) 
tauff = lambda Te,nu , EM : 0.05468 *   (Te)**-1.5 * (nu)**-2 * EM *gff (Te,nu) 
Zi =1 
gff =lambda Te ,nu :  np.log (np.exp(1.0) +np.exp (5.960 -np.sqrt(3)/np.pi *np.log (Zi *nu *(Te/(1e4)  )**-1.5  )) )  

pysm_sync_mod = 's7'

In [ ]:
# eda map is at 159 Mhz and 3.1 degree
goal_f = 159.0
fstring = "EDA2_159MHz_I_noPrior_HPX2D.fits"
try :
    print(f"reading {fstring} ")
    edamap =  hp.read_map(filename=f"../input_data/{fstring}")  
except FileNotFoundError: 
    filename = wget.download( f"https://lambda.gsfc.nasa.gov/data/foregrounds/EDA2/{fstring}" , out ="../input_data")
    edamap =  hp.read_map(f"../input_data/{fstring}")
eda2 = hp.ud_grade(edamap, nside)
eda2 = rotate_map(eda2, 'C', 'G')
thevec = hp.pixelfunc.ang2vec(np.pi/2.7, np.pi/1.5, lonlat=False)
centre = hp.query_disc(nside, thevec, 0.7)
eda2[centre] = np.nan

eda2[np.where(np.isnan(eda2))[0]] = hp.UNSEEN
eda2_5 = hp.smoothing(eda2, fwhm=np.radians(np.sqrt((5.0)**2 - (3.1)**2)))
has_5 = hp.smoothing(hasmap, fwhm=np.radians(np.sqrt((5.0)**2 - (56./60.)**2)))
haserr = np.sqrt((0.1 * hasmap)**2 + 1.3**2) /  hasmap
eda2_5[np.where(eda2_5 == hp.UNSEEN)] = np.nan
eda2[np.where(eda2 == hp.UNSEEN)] = np.nan
eda2_off = remove_off(eda2_5, has_5, 0.05, haserr)

hp.mollview(eda2 - eda2_off, min=130, max=1000)
plt.title('EDA2')
plt.show()

beta_159 = (beta_map + cs_map * np.log(goal_f/45.))
ff159 = Tff(Te=planck_te, nu=goal_f/1000., EM=hust)
param_159 = hasmap * (goal_f/408.)**beta_159 + hp.smoothing(ff159, fwhm=np.radians(np.sqrt((3.1)**2 - 1.0**2)))
param_159[np.where(np.isnan(param_159))[0]] = hp.UNSEEN
param_159 = hp.smoothing(param_159, fwhm=np.radians(np.sqrt((3.1)**2 - (56./60.)**2)))
param_159[np.where(param_159 == hp.UNSEEN)[0]] = np.nan
param_159[np.where(np.isnan(eda2))[0]] = np.nan

hp.mollview(param_159,  min=130, max=1000)
plt.title('Parametric 159 model')
plt.show()

sky = pysm.Sky(nside=nside, preset_strings=[pysm_sync_mod])
pysm159 = sky.get_emission((goal_f/1000.) * u.GHz)[0] * 1.e-6 #gives Stokes I, Q and U in microK
pysm159.value[np.where(np.isnan(eda2))[0]] = hp.UNSEEN
map_resol = hp.nside2resol(nside, arcmin=True) / 60.
pysm159 = hp.smoothing(pysm159.value, fwhm=np.radians(np.sqrt((3.1)**2 - map_resol**2))) + hp.smoothing(ff159, fwhm=np.radians(np.sqrt((3.1)**2 - 1.0**2)))
pysm159[np.where(pysm159 == hp.UNSEEN)] = np.nan

hp.mollview(pysm159,  min=130, max=1000)
plt.title('PSM 159')
plt.show()

# galactic plane mask
fstring = f"HFI_Mask_GalPlane-apo0_2048_R2.00.fits"
try : 
    print(f"reading {fstring} ")
    gmask = hp.read_map(f"/input_data/{fstring}" , field=4)
except FileNotFoundError: 
    filename = wget.download( f"http://pla.esac.esa.int/pla/aio/product-action?SOURCE_LIST.NAME={fstring}", out ="../input_data") 
    gmask = hp.read_map(f"/input_data/{fstring}" , field=4)  

gmask = hp.ud_grade(gmask, nside)
gmask = np.where(gmask < 0.99, 0.0, 1.1)
gmask[np.where(gmask == 0.0)[0]] = np.nan   

diff = eda2 - eda2_off - param_159
diff_pysm = eda2 - eda2_off - pysm159

cmapEC = cmr.eclipse  

fig, ((ax1, ax2)) = plt.subplots(nrows=2, ncols=1, figsize=(6,8))
plt.axes(ax1)
hp.mollview(diff / (eda2 - eda2_off),  min=-1.0, max=1.0, hold=True, title='Param 159 MHz', cmap=cmapEC)
plt.axes(ax2)
hp.mollview(diff_pysm / (eda2 - eda2_off),  min=-1.0, max=1.0, hold=True, title='PySM 159 MHz', cmap=cmapEC)
plt.show()

scat_x = (eda2 - eda2_off) * gmask
notnans = np.where(~np.isnan(scat_x))[0]

fig, ((ax1, ax2, ax3)) = plt.subplots(nrows=1, ncols=3, figsize=(24,6))
plt.axes(ax1)
plt.scatter(scat_x, pysm159* gmask, s=20, edgecolors='teal', facecolors='none', label='PYSM3 Model', alpha=0.4)
plt.scatter(scat_x, param_159* gmask, s=20, edgecolors='darkslateblue', facecolors='none', label='Parametric Model', alpha=0.4)
ax1.tick_params(axis='both', which='major', labelsize=14)
ax1.tick_params(axis='both', which='minor', labelsize=14)
plt.xlim([0.0, np.nanmax(pysm159*gmask)+200.])
plt.ylim([0.0, np.nanmax(pysm159*gmask)+200.])
plt.plot(scat_x, scat_x, 'k-')
plt.text(870, 2700, f'Parametric corr: {np.corrcoef(scat_x[notnans], (param_159* gmask)[notnans])[0,1]:.3f} K', color='darkslateblue', fontsize=14)
plt.text(870, 2400, f'PySM corr: {np.corrcoef(scat_x[notnans], (pysm159* gmask)[notnans])[0,1]:.3f} K', color='teal', fontsize=14)
plt.xlabel('Data 159 MHz Brightness Temperature (K)', fontsize=14)
plt.ylabel('Model 159 MHz Brightness Temperature (K)', fontsize=14)
plt.axes(ax2)
plt.hist(abs(diff/(eda2 - eda2_off))*100., bins=40, range=[0, 100], label='Parametric (Full)', histtype='step', color='darkslateblue')
plt.hist(abs(diff_pysm/(eda2 - eda2_off))*100., bins=40, range=[0, 100], label='PYSM3 (Full)', histtype='step', color='teal')
plt.axvline(x=np.nanmedian(abs(diff/(eda2 - eda2_off))*100.), color='darkslateblue', linestyle='--')
plt.axvline(x=np.nanmedian(abs(diff_pysm/(eda2 - eda2_off))*100.), color='teal', linestyle='--')
ax2.tick_params(axis='both', which='major', labelsize=14)
ax2.tick_params(axis='both', which='minor', labelsize=14)
plt.xlabel('Absolute Percentage Difference (%)', fontsize=14)
plt.ylabel('Number of Pixels', fontsize=14)
plt.legend(fontsize=12)
plt.axes(ax3)
plt.hist(abs(diff/(eda2 - eda2_off))*100.*gmask, bins=40, range=[0, 100], label='Parametric (NoGal)', histtype='step', color='darkslateblue')
plt.hist(abs(diff_pysm/(eda2 - eda2_off))*100.*gmask, bins=40, range=[0, 100], label='PYSM3 (NoGal)', histtype='step', color='teal')
plt.axvline(x=np.nanmedian(abs(diff/(eda2 - eda2_off))*100.*gmask), color='darkslateblue', linestyle='--')
plt.axvline(x=np.nanmedian(abs(diff_pysm/(eda2 - eda2_off))*100.*gmask), color='teal', linestyle='--')
ax3.tick_params(axis='both', which='major', labelsize=14)
ax3.tick_params(axis='both', which='minor', labelsize=14)
plt.xlabel('Absolute Percentage Difference (%)', fontsize=14)
plt.ylabel('Number of Pixels', fontsize=14)
plt.legend(fontsize=12)
plt.show()

print(np.nanmedian(abs(diff/(eda2 - eda2_off))*100.*gmask), np.nanpercentile(abs(diff/(eda2 - eda2_off))*100.*gmask, 25), np.nanpercentile(abs(diff/(eda2 - eda2_off))*100.*gmask, 75))
print(np.nanmedian(abs(diff_pysm/(eda2 - eda2_off))*100.*gmask), np.nanpercentile(abs(diff_pysm/(eda2 - eda2_off))*100.*gmask, 25), np.nanpercentile(abs(diff_pysm/(eda2 - eda2_off))*100.*gmask, 75))

In [ ]:
# dwingoloo is at 820 Mhz and 1.2 degree
goal_f = 820.0
fstring = "Dwingeloo_Kelvins_1_256.fits"
try :
    print(f"reading {fstring} ")
    dwing =  hp.read_map(filename=f"../input_data/{fstring}")  
except FileNotFoundError: 
    filename = wget.download( f"https://lambda.gsfc.nasa.gov/data/foregrounds/dwingeloo_820/{fstring}" , out ="../input_data")
    dwing =  hp.read_map(f"../input_data/{fstring}")
dwing[np.where(dwing < -1000)[0]] = np.nan
dwing[np.where(np.isnan(dwing))[0]] = hp.UNSEEN
dwing_5 = hp.smoothing(dwing, fwhm=np.radians(np.sqrt((5.0)**2 - (1.2)**2)))
dwing_5[np.where(dwing_5 == hp.UNSEEN)] = np.nan
dwing[np.where(dwing == hp.UNSEEN)] = np.nan
dwing_off = remove_off(dwing_5, has_5, 0.06, haserr)

hp.mollview(dwing, norm='hist')
plt.title('Dwingeloo')
plt.show()

beta_820 = (beta_map + cs_map * np.log(goal_f/45.))
ff820 = Tff(Te=planck_te, nu=goal_f/1000., EM=hust)
param_820 = hasmap * (goal_f/408.)**beta_820 + hp.smoothing(ff820, fwhm=np.radians(np.sqrt((1.2)**2 - 1.0**2)))
param_820[np.where(np.isnan(param_820))[0]] = hp.UNSEEN
param_820 = hp.smoothing(param_820, fwhm=np.radians(np.sqrt((1.2)**2 - (56./60.)**2)))
param_820[np.where(param_820 == hp.UNSEEN)[0]] = np.nan
param_820[np.where(np.isnan(dwing))] = np.nan

hp.mollview(param_820, norm='hist')
plt.title('Parametric 820 model')
plt.show()

ff820 = Tff(Te=planck_te, nu=goal_f/1000., EM=hust)
sky = pysm.Sky(nside=nside, preset_strings=[pysm_sync_mod])
pysm820 = sky.get_emission((goal_f/1000.) * u.GHz)[0] * 1.e-6 #gives Stokes I, Q and U in microK
pysm820.value[np.where(np.isnan(dwing))[0]] = hp.UNSEEN
map_resol = hp.nside2resol(nside, arcmin=True) / 60.
pysm820 = hp.smoothing(pysm820.value, fwhm=np.radians(np.sqrt((1.2)**2 - map_resol**2))) + hp.smoothing(ff820, fwhm=np.radians(np.sqrt((1.2)**2 - 1.0**2)))
pysm820[np.where(pysm820 == hp.UNSEEN)] = np.nan

hp.mollview(pysm820, norm='hist')
plt.title('PYSM 820')
plt.show()

diff = dwing - dwing_off - param_820
diff_pysm = dwing - dwing_off - pysm820

fig, ((ax1, ax2)) = plt.subplots(nrows=2, ncols=1, figsize=(6,8))
plt.axes(ax1)
hp.mollview(diff / (dwing - dwing_off),  min=-1.0, max=1.0, hold=True, title='Param 820 MHz', cmap=cmapEC)
plt.axes(ax2)
hp.mollview(diff_pysm / (dwing - dwing_off ),  min=-1.0, max=1.0, hold=True, title='PySM 820 MHz', cmap=cmapEC)
plt.show()

scat_x = (dwing - dwing_off) * gmask
notnans = np.where(~np.isnan(scat_x))[0]

fig, ((ax1, ax2, ax3)) = plt.subplots(nrows=1, ncols=3, figsize=(24,6))
plt.axes(ax1)
plt.scatter(scat_x, pysm820*gmask, s=20, edgecolors='teal', facecolors='none', label='PYSM3 Model', alpha=0.4)
plt.scatter(scat_x, param_820*gmask, s=20, edgecolors='darkslateblue', facecolors='none', label='Parametric Model', alpha=0.4)
plt.plot(scat_x, scat_x, 'k-')
plt.xlim([0.0, 1+np.nanmax(pysm820*gmask)])
plt.ylim([0.0, 1+np.nanmax(pysm820*gmask)])
ax1.tick_params(axis='both', which='major', labelsize=14)
ax1.tick_params(axis='both', which='minor', labelsize=14)
plt.text(1, 10, f'Parametric corr: {np.corrcoef(scat_x[notnans], (param_820*gmask)[notnans])[0,1]:.3f} K', color='darkslateblue', fontsize=14)
plt.text(1, 8.5, f'PYSM corr: {np.corrcoef(scat_x[notnans], (pysm820*gmask)[notnans])[0,1]:.3f} K', color='teal', fontsize=14)
plt.xlabel('Data 820 MHz Brightness Temperature (K)', fontsize=14)
plt.ylabel('Model 820 MHz Brightness Temperature (K)', fontsize=14)
plt.axes(ax2)
plt.hist(abs(diff/(dwing - dwing_off))*100., bins=40, range=[0, 300], label='Parametric (Full)', histtype='step', color='darkslateblue')
plt.hist(abs(diff_pysm/(dwing - dwing_off))*100., bins=40, range=[0, 200], label='PYSM3 (Full)', histtype='step', color='teal')
plt.axvline(x=np.nanmedian(abs(diff/(dwing - dwing_off))*100.), color='darkslateblue', linestyle='--')
plt.axvline(x=np.nanmedian(abs(diff_pysm/(dwing - dwing_off))*100.), color='teal', linestyle='--')
ax2.tick_params(axis='both', which='major', labelsize=14)
ax2.tick_params(axis='both', which='minor', labelsize=14)
plt.xlabel('Absolute Percentage Difference (%)', fontsize=14)
plt.ylabel('Number of Pixels', fontsize=14)
plt.legend(fontsize=12)
plt.axes(ax3)
plt.hist(abs(diff/(dwing - dwing_off))*100.*gmask, bins=40, range=[0, 300], label='Parametric (No Gal)', histtype='step', color='darkslateblue')
plt.hist(abs(diff_pysm/(dwing - dwing_off))*100.*gmask, bins=40, range=[0, 200], label='PYSM3 (No Gal)', histtype='step', color='teal')
plt.axvline(x=np.nanmedian(abs(diff/(dwing - dwing_off))*100.*gmask), color='darkslateblue', linestyle='--')
plt.axvline(x=np.nanmedian(abs(diff_pysm/(dwing - dwing_off))*100.*gmask), color='teal', linestyle='--')
ax3.tick_params(axis='both', which='major', labelsize=14)
ax3.tick_params(axis='both', which='minor', labelsize=14)
plt.xlabel('Absolute Percentage Difference (%)', fontsize=14)
plt.ylabel('Number of Pixels', fontsize=14)
plt.legend(fontsize=12)
plt.show()

print(np.nanmedian(abs(diff/(dwing - dwing_off))*100.*gmask), np.nanpercentile(abs(diff/(dwing - dwing_off))*100.*gmask, 25), np.nanpercentile(abs(diff/(dwing - dwing_off))*100.*gmask, 75))
print(np.nanmedian(abs(diff_pysm/(dwing - dwing_off))*100.*gmask), np.nanpercentile(abs(diff_pysm/(dwing - dwing_off))*100.*gmask, 25), np.nanpercentile(abs(diff_pysm/(dwing - dwing_off))*100.*gmask, 75))

In [ ]:
# jonas is at 2326 Mhz and 20 arcmin
goal_f = 2326.0
fstring = "lambda_23de_hea.fits"
try :
    print(f"reading {fstring} ")
    jonas =  hp.read_map(filename=f"../input_data/{fstring}")  
except FileNotFoundError: 
    filename = wget.download( f"https://lambda.gsfc.nasa.gov/data/foregrounds/rhodes_2326/{fstring}" , out ="../input_data")
    jonas =  hp.read_map(f"../input_data/{fstring}")
jonas = rotate_map(jonas, 'C', 'G')
jonas[np.where(jonas < 0.03)[0]] = np.nan
jonas[np.where(np.isnan(jonas))[0]] = hp.UNSEEN
jonas_5 = hp.smoothing(jonas, fwhm=np.radians(np.sqrt((5.0)**2 - (20./60.)**2)))
jonas_5[np.where(jonas_5 == hp.UNSEEN)] = np.nan
jonas = hp.smoothing(jonas, fwhm=np.radians(np.sqrt((56./60.)**2 - (20./60.)**2)))
jonas[np.where(jonas == hp.UNSEEN)] = np.nan
jonas_off = remove_off(jonas_5, has_5, 0.06, haserr)

hp.mollview(jonas, norm='hist')
plt.title('Jonas')
plt.show()

beta_2326 = (beta_map + cs_map * np.log(goal_f/45.))
ff2326 = Tff(Te=planck_te, nu=goal_f/1000., EM=hust)
param_2326 = hasmap * (goal_f/408.)**beta_2326 + ff2326
param_2326[np.where(np.isnan(param_2326))[0]] = hp.UNSEEN
param_2326[np.where(param_2326 == hp.UNSEEN)[0]] = np.nan
param_2326[np.where(np.isnan(jonas))] = np.nan

hp.mollview(param_2326, norm='hist')
plt.title('Parametric 2326 model')
plt.show()

ff2326 = Tff(Te=planck_te, nu=goal_f/1000., EM=hust)
sky = pysm.Sky(nside=nside, preset_strings=[pysm_sync_mod])
pysm2326 = sky.get_emission((goal_f/1000.) * u.GHz)[0] * 1.e-6 #gives Stokes I, Q and U in microK
pysm2326.value[np.where(np.isnan(jonas))[0]] = hp.UNSEEN
map_resol = hp.nside2resol(nside, arcmin=True) / 60.
pysm2326 = hp.smoothing(pysm2326.value, fwhm=np.radians(np.sqrt((56./60.)**2 - map_resol**2))) + ff2326
pysm2326[np.where(pysm2326 == hp.UNSEEN)] = np.nan

hp.mollview(pysm2326, norm='hist')
plt.title('PYSM 2326')
plt.show()

diff = jonas - jonas_off - param_2326
diff_pysm = jonas - jonas_off - pysm2326

fig, ((ax1, ax2)) = plt.subplots(nrows=2, ncols=1, figsize=(6,8))
plt.axes(ax1)
hp.mollview(diff / (jonas - jonas_off),  min=-1.0, max=1.0, hold=True, title='Param 2326 MHz', cmap=cmapEC)
plt.axes(ax2)
hp.mollview(diff_pysm / (jonas - jonas_off),  min=-1.0, max=1.0, hold=True, title='PySM 2326 MHz', cmap=cmapEC)
plt.show()

scat_x = (jonas - jonas_off) * gmask
notnans = np.where(~np.isnan(scat_x))[0]

fig, ((ax1, ax2, ax3)) = plt.subplots(nrows=1, ncols=3, figsize=(24,6))
plt.axes(ax1)
plt.scatter(scat_x, pysm2326*gmask, s=20, edgecolors='teal', facecolors='none', label='PYSM3 Model', alpha=0.4)
plt.scatter(scat_x, param_2326*gmask, s=20, edgecolors='darkslateblue', facecolors='none', label='Parametric Model', alpha=0.4)
plt.plot(scat_x, scat_x, 'k-')
plt.xlim([0.0, 1+np.nanmax(param_2326*gmask)])
plt.ylim([0.0, 1+np.nanmax(param_2326*gmask)])
ax1.tick_params(axis='both', which='major', labelsize=14)
ax1.tick_params(axis='both', which='minor', labelsize=14)
plt.text(4.5, 8.1, f'PYSM corr: {np.corrcoef(scat_x [notnans], (pysm2326*gmask)[notnans])[0,1]:.3f} K', color='teal', fontsize=14)
plt.text(4.5, 7.4, f'Parametric corr: {np.corrcoef(scat_x [notnans], (param_2326*gmask)[notnans])[0,1]:.3f} K', color='darkslateblue', fontsize=14)
plt.xlabel('Data 2326 MHz Brightness Temperature (K)', fontsize=14)
plt.ylabel('Model 2326 MHz Brightness Temperature (K)', fontsize=14)
plt.axes(ax2)
plt.hist(abs(diff/(jonas - jonas_off))*100., bins=40, range=[0, 120], label='Parametric (Full)', histtype='step', color='darkslateblue')
plt.hist(abs(diff_pysm/(jonas - jonas_off))*100., bins=40, range=[0, 120], label='PYSM3 (Full)', histtype='step', color='teal')
plt.axvline(x=np.nanmedian(abs(diff/(jonas - jonas_off))*100.), color='darkslateblue', linestyle='--')
plt.axvline(x=np.nanmedian(abs(diff_pysm/(jonas - jonas_off))*100.), color='teal', linestyle='--')
ax2.tick_params(axis='both', which='major', labelsize=14)
ax2.tick_params(axis='both', which='minor', labelsize=14)
plt.xlabel('Absolute Percentage Difference (%)', fontsize=14)
plt.ylabel('Number of Pixels', fontsize=14)
plt.legend(fontsize=12)
plt.axes(ax3)
plt.hist(abs(diff/(jonas - jonas_off))*100., bins=40, range=[0, 120], label='Parametric (NoGal)', histtype='step', color='darkslateblue')
plt.hist(abs(diff_pysm/(jonas - jonas_off))*100.*gmask, bins=40, range=[0, 120], label='PYSM3  (NoGal)', histtype='step', color='teal')
plt.axvline(x=np.nanmedian(abs(diff/(jonas - jonas_off))*100.*gmask), color='darkslateblue', linestyle='--')
plt.axvline(x=np.nanmedian(abs(diff_pysm/(jonas - jonas_off))*100*gmask), color='teal', linestyle='--')
ax3.tick_params(axis='both', which='major', labelsize=14)
ax3.tick_params(axis='both', which='minor', labelsize=14)
plt.xlabel('Absolute Percentage Difference (%)', fontsize=14)
plt.ylabel('Number of Pixels', fontsize=14)
plt.legend(fontsize=12)
plt.show()

print(np.nanmedian(abs(diff/(jonas - jonas_off))*100.*gmask), np.nanpercentile(abs(diff/(jonas - jonas_off))*100.*gmask, 25), np.nanpercentile(abs(diff/(jonas - jonas_off))*100.*gmask, 75))
print(np.nanmedian(abs(diff_pysm/(jonas - jonas_off))*100.*gmask), np.nanpercentile(abs(diff_pysm/(jonas - jonas_off))*100.*gmask, 25), np.nanpercentile(abs(diff_pysm/(jonas - jonas_off))*100.*gmask, 75))

In [ ]:
# spass is at 2303 Mhz and 8.9 arcmin
goal_f = 2303.0
fstring = "spass_dr1_1902_healpix_Tb.i.fits"
try :
    print(f"reading {fstring} ")
    spass =  hp.read_map(filename=f"../input_data/{fstring}")  
except FileNotFoundError: 
    filename = wget.download( f"https://lambda.gsfc.nasa.gov/data/foregrounds/spass/{fstring}" , out ="../input_data")
    spass =  hp.read_map(f"../input_data/{fstring}")
spass = hp.ud_grade(spass, 256)
spass[np.where(spass < -1000)] = np.nan
spass[np.where(np.isnan(spass))[0]] = hp.UNSEEN
spass_5 = hp.smoothing(spass, fwhm=np.radians(np.sqrt((5.0)**2 - (8.9/60.)**2)))
spass_5[np.where(spass_5 == hp.UNSEEN)] = np.nan
spass= hp.smoothing(spass, fwhm=np.radians(np.sqrt((56./60.)**2 - (8.9/60.)**2)))
spass[np.where(spass == hp.UNSEEN)] = np.nan
spass_off = remove_off(spass_5, has_5, 0.05, haserr)

hp.mollview(spass, norm='hist')
plt.title('SPASS')
plt.show()

beta_2303 = (beta_map + cs_map * np.log(goal_f/45.))
ff2303 = Tff(Te=planck_te, nu=goal_f/1000., EM=hust)
param_2303 = hasmap * (goal_f/408.)**beta_2303 + ff2303
param_2303[np.where(np.isnan(param_2303))[0]] = hp.UNSEEN
param_2303[np.where(param_2303 == hp.UNSEEN)[0]] = np.nan
param_2303[np.where(np.isnan(spass))] = np.nan

hp.mollview(param_2303, norm='hist')
plt.title('Parametric 2303 model')
plt.show()

sky = pysm.Sky(nside=nside, preset_strings=[pysm_sync_mod])
pysm2303= sky.get_emission((goal_f/1000.) * u.GHz)[0] * 1.e-6 #gives Stokes I, Q and U in microK
pysm2303.value[np.where(np.isnan(spass))[0]] = hp.UNSEEN
map_resol = hp.nside2resol(nside, arcmin=True) / 60.
pysm2303 = hp.smoothing(pysm2303.value, fwhm=np.radians(np.sqrt((56./60.)**2 - map_resol**2))) + ff2303
pysm2303[np.where(pysm2303 == hp.UNSEEN)] = np.nan

hp.mollview(pysm2303, norm='hist')
plt.title('PYSM 2303')
plt.show()

diff = spass - spass_off - param_2303
diff_pysm = spass - spass_off - pysm2303

fig, ((ax1, ax2)) = plt.subplots(nrows=2, ncols=1, figsize=(6,8))
plt.axes(ax1)
hp.mollview(diff / (spass - spass_off),  min=-1.0, max=1.0, hold=True, title='Param 2303 MHz', cmap=cmapEC)
plt.axes(ax2)
hp.mollview(diff_pysm / (spass - spass_off),  min=-1.0, max=1.0, hold=True, title='PySM 2303 MHz', cmap=cmapEC)
plt.show()


scat_x = (spass - spass_off) * gmask
notnans = np.where(~np.isnan(scat_x))[0]

fig, ((ax1, ax2, ax3)) = plt.subplots(nrows=1, ncols=3, figsize=(24,6))
plt.axes(ax1)
plt.scatter(scat_x, pysm2303*gmask, s=20, edgecolors='teal', facecolors='none', label='PYSM3 Model', alpha=0.4)
plt.scatter(scat_x , param_2303*gmask, s=20, edgecolors='darkslateblue', facecolors='none', label='Parametric Model', alpha=0.4)
plt.plot(scat_x, scat_x, 'k-')
plt.xlim([0.0, 1+np.nanmax(pysm2303*gmask)])
plt.ylim([0.0, 1+np.nanmax(pysm2303*gmask)])
ax1.tick_params(axis='both', which='major', labelsize=14)
ax1.tick_params(axis='both', which='minor', labelsize=14)
plt.text(0.2, 1.76, f'Parametric corr: {np.corrcoef(scat_x[notnans], (param_2303*gmask)[notnans])[0,1]:.3f} K', color='darkslateblue', fontsize=14)
plt.text(0.2, 1.46, f'PYSM corr: {np.corrcoef(scat_x[notnans], (pysm2303*gmask)[notnans])[0,1]:.3f} K', color='teal', fontsize=14)
plt.xlabel('Data 2303 MHz Brightness Temperature (K)', fontsize=14)
plt.ylabel('Model 2303 MHz Brightness Temperature (K)', fontsize=14)
plt.axes(ax2)
plt.hist(abs(diff/(spass - spass_off))*100., bins=40, range=[0, 120], label='Parametric (Full)', histtype='step', color='darkslateblue')
plt.hist(abs(diff_pysm/(spass - spass_off))*100., bins=40, range=[0, 120], label='PYSM3 (Full)', histtype='step', color='teal')
plt.axvline(x=np.nanmedian(abs(diff/(spass - spass_off))*100.), color='darkslateblue', linestyle='--')
plt.axvline(x=np.nanmedian(abs(diff_pysm/(spass - spass_off))*100.), color='teal', linestyle='--')
ax2.tick_params(axis='both', which='major', labelsize=14)
ax2.tick_params(axis='both', which='minor', labelsize=14)
plt.xlabel('Absolute Percentage Difference (%)', fontsize=14)
plt.ylabel('Number of Pixels', fontsize=14)
plt.legend(fontsize=12, loc='upper right')
plt.axes(ax3)
plt.hist(abs(diff/(spass - spass_off))*100.*gmask, bins=40, range=[0, 120], label='Parametric (NoGal)', histtype='step', color='darkslateblue')
plt.hist(abs(diff_pysm/(spass - spass_off))*100.*gmask, bins=40, range=[0, 120], label='PYSM3 (NoGal)', histtype='step', color='teal')
plt.axvline(x=np.nanmedian(abs(diff/(spass - spass_off))*100.*gmask), color='darkslateblue', linestyle='--')
plt.axvline(x=np.nanmedian(abs(diff_pysm/(spass - spass_off))*100.*gmask), color='teal', linestyle='--')
ax3.tick_params(axis='both', which='major', labelsize=14)
ax3.tick_params(axis='both', which='minor', labelsize=14)
plt.xlabel('Absolute Percentage Difference (%)', fontsize=14)
plt.ylabel('Number of Pixels', fontsize=14)
plt.legend(fontsize=12, loc='upper right')
plt.show()

print(np.nanmedian(abs(diff/(spass - spass_off))*100.*gmask), np.nanpercentile(abs(diff/(spass - spass_off))*100.*gmask, 25), np.nanpercentile(abs(diff/(spass - spass_off))*100.*gmask, 75))
print(np.nanmedian(abs(diff_pysm/(spass - spass_off))*100.*gmask), np.nanpercentile(abs(diff_pysm/(spass - spass_off))*100.*gmask, 25), np.nanpercentile(abs(diff_pysm/(spass - spass_off))*100.*gmask, 75))

In [ ]:
# q11 is at 11000 Mhz and 1 deg
goal_f = 11000.0
fstring = "quijote_mfi_smth_skymap_11ghz_512_dr1.fits"
try :
    print(f"reading {fstring} ")
    q11 =  hp.read_map(filename=f"../input_data/{fstring}")  
except FileNotFoundError: 
    filename = wget.download( f"https://lambda.gsfc.nasa.gov/data/suborbital/QUIJOTE/QUIJOTE-MFI-Release1/{fstring}" , out ="../input_data")
    q11 =  hp.read_map(f"../input_data/{fstring}")
q11 = hp.ud_grade(q11, 256) #convert from mk to K
q11_5 = hp.smoothing(q11, fwhm=np.radians(np.sqrt((5.0)**2 - (60./60.)**2)))
q11_5[np.where(q11_5 == hp.UNSEEN)] = np.nan
q11[np.where(q11 == hp.UNSEEN)] = np.nan
q11_off = remove_off(q11_5, has_5, 0.05, haserr)
q11 = q11 / 1000.
q11_off= q11_off / 1000.

hp.mollview(q11, norm='hist')
plt.title('q11')
plt.show()

beta_11 = (beta_map + cs_map * np.log(goal_f/45.))
ff11 = Tff(Te=planck_te, nu=goal_f/1000., EM=hust)
param_11 = hasmap * (goal_f/408.)**beta_11 + ff11
param_11[np.where(np.isnan(param_11))[0]] = hp.UNSEEN
param_11[np.where(param_11 == hp.UNSEEN)[0]] = np.nan
param_11[np.where(np.isnan(q11))] = np.nan

hp.mollview(param_11, norm='hist')
plt.title('Parametric 11000 model')
plt.show()

sky = pysm.Sky(nside=nside, preset_strings=[pysm_sync_mod])
pysm11= sky.get_emission((goal_f/1000.) * u.GHz)[0] * 1.e-6 #gives Stokes I, Q and U in microK
pysm11.value[np.where(np.isnan(q11))[0]] = hp.UNSEEN
map_resol = hp.nside2resol(nside, arcmin=True) / 60.
pysm11 = hp.smoothing(pysm11.value, fwhm=np.radians(np.sqrt((56./60.)**2 - map_resol**2))) + ff11
pysm11[np.where(pysm11 == hp.UNSEEN)] = np.nan

hp.mollview(pysm11, norm='hist')
plt.title('PYSM 11000')
plt.show()


diff = q11 - q11_off - param_11
diff_pysm = q11 - q11_off - pysm11

fig, ((ax1, ax2)) = plt.subplots(nrows=2, ncols=1, figsize=(6,8))
plt.axes(ax1)
hp.mollview(diff / (q11 - q11_off),  min=-1.0, max=1.0, hold=True, title='Param 11000 MHz', cmap=cmapEC)
plt.axes(ax2)
hp.mollview(diff_pysm / (q11 - q11_off),  min=-1.0, max=1.0, hold=True, title='PySM 11000 MHz', cmap=cmapEC)
plt.show()


scat_x = (q11 - q11_off) * gmask
notnans = np.where(~np.isnan(scat_x))[0]

fig, ((ax1, ax2, ax3)) = plt.subplots(nrows=1, ncols=3, figsize=(24,6))
plt.axes(ax1)
plt.scatter(scat_x , param_11*gmask, s=20, edgecolors='darkslateblue', facecolors='none', label='Parametric Model')
plt.scatter(scat_x, pysm11*gmask, s=20, edgecolors='teal', facecolors='none', label='PYSM3 Model')
plt.plot(scat_x, scat_x, 'k-')
plt.xlim([0.0, 0.03+np.nanmax(pysm11*gmask)])
plt.ylim([0.0, 0.03+np.nanmax(pysm11*gmask)])
ax1.tick_params(axis='both', which='major', labelsize=14)
ax1.tick_params(axis='both', which='minor', labelsize=14)
plt.text(0.02, 0.04, f'Parametric corr: {np.corrcoef(scat_x[notnans], (param_11*gmask)[notnans])[0,1]:.3f} K', color='darkslateblue', fontsize=14)
plt.text(0.02, 0.035, f'PYSM corr: {np.corrcoef(scat_x[notnans], (pysm11*gmask)[notnans])[0,1]:.3f} K', color='teal', fontsize=14)
plt.xlabel('Data 11000 MHz Brightness Temperature (K)', fontsize=14)
plt.ylabel('Model 11000 MHz Brightness Temperature (K)', fontsize=14)
plt.axes(ax2)
plt.hist(abs(diff/(q11 - q11_off))*100., bins=40, range=[0, 150], label='Parametric (Full)', histtype='step', color='darkslateblue')
plt.hist(abs(diff_pysm/(q11 - q11_off))*100., bins=40, range=[0, 150], label='PYSM3 (Full)', histtype='step', color='teal')
plt.axvline(x=np.nanmedian(abs(diff/(q11 - q11_off))*100.), color='darkslateblue', linestyle='--')
plt.axvline(x=np.nanmedian(abs(diff_pysm/(q11 - q11_off))*100.), color='teal', linestyle='--')
ax2.tick_params(axis='both', which='major', labelsize=14)
ax2.tick_params(axis='both', which='minor', labelsize=14)
plt.xlabel('Absolute Percentage Difference (%)', fontsize=14)
plt.ylabel('Number of Pixels', fontsize=14)
plt.legend(fontsize=12, loc='upper right')
plt.axes(ax3)
plt.hist(abs(diff/(q11 - q11_off))*100.*gmask, bins=40, range=[0, 150], label='Parametric (NoGal)', histtype='step', color='darkslateblue')
plt.hist(abs(diff_pysm/(q11 - q11_off))*100.*gmask, bins=40, range=[0, 150], label='PYSM3 (NoGal)', histtype='step', color='teal')
plt.axvline(x=np.nanmedian(abs(diff/(q11 - q11_off))*100.*gmask), color='darkslateblue', linestyle='--')
plt.axvline(x=np.nanmedian(abs(diff_pysm/(q11 - q11_off))*100.*gmask), color='teal', linestyle='--')
ax3.tick_params(axis='both', which='major', labelsize=14)
ax3.tick_params(axis='both', which='minor', labelsize=14)
plt.xlabel('Absolute Percentage Difference (%)', fontsize=14)
plt.ylabel('Number of Pixels', fontsize=14)
plt.legend(fontsize=12, loc='upper right')
plt.show()

print(np.nanmedian(abs(diff/(q11 - q11_off))*100.*gmask), np.nanpercentile(abs(diff/(q11 - q11_off))*100.*gmask, 25), np.nanpercentile(abs(diff/(q11 - q11_off))*100.*gmask, 75))
print(np.nanmedian(abs(diff_pysm/(q11 - q11_off))*100.*gmask), np.nanpercentile(abs(diff_pysm/(q11 - q11_off))*100.*gmask, 25), np.nanpercentile(abs(diff_pysm/(q11 - q11_off))*100.*gmask, 75))